# Introduction to Invariant Extended Kalman Filter

This notebook provides a gentle introduction to the Invariant Extended Kalman Filter (IEKF) through a simple attitude estimation example.

## Learning Objectives

By the end of this notebook, you will:
- Understand the difference between standard EKF and IEKF
- Learn about Lie groups and their application to state estimation
- Implement a simple IEKF for attitude estimation
- Compare IEKF performance with standard EKF

## Prerequisites

- Basic understanding of Kalman filtering
- Familiarity with Python and NumPy
- Knowledge of rotation matrices (helpful but not required)

In [ ]:
# Install required packages (uncomment if needed)
# !pip install numpy scipy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
from scipy.linalg import expm, logm

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Lie Groups and SO(3)

SO(3) is the Special Orthogonal group of 3D rotations. It's a Lie group with the following properties:

- **Elements**: 3×3 rotation matrices R where R^T R = I and det(R) = 1
- **Group operation**: Matrix multiplication
- **Tangent space**: The Lie algebra so(3) of 3×3 skew-symmetric matrices
- **Exponential map**: Maps from so(3) to SO(3)
- **Logarithm map**: Maps from SO(3) to so(3)

In [ ]:
def skew(v):
    """Convert vector to skew-symmetric matrix."""
    return np.array([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])

def unskew(S):
    """Convert skew-symmetric matrix to vector."""
    return np.array([S[2,1], S[0,2], S[1,0]])

def exp_so3(omega):
    """Exponential map from so(3) to SO(3)."""
    angle = np.linalg.norm(omega)
    if angle < 1e-10:
        return np.eye(3)
    axis = omega / angle
    K = skew(axis)
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * K @ K

def log_so3(R):
    """Logarithm map from SO(3) to so(3)."""
    angle = np.arccos((np.trace(R) - 1) / 2)
    if angle < 1e-10:
        return np.zeros(3)
    return angle / (2 * np.sin(angle)) * unskew(R - R.T)

# Test the functions
omega_test = np.array([0.1, 0.2, 0.3])
R_test = exp_so3(omega_test)
omega_recovered = log_so3(R_test)

print(f"Original omega: {omega_test}")
print(f"Recovered omega: {omega_recovered}")
print(f"Error: {np.linalg.norm(omega_test - omega_recovered)}")

## 2. Attitude Estimation Problem

We'll simulate an attitude estimation problem:

- **State**: Rotation matrix R ∈ SO(3)
- **Dynamics**: R_dot = R * skew(omega), where omega is angular velocity
- **Measurement**: Noisy vectors in body frame (e.g., from accelerometer, magnetometer)

### True System Simulation

In [ ]:
# Simulation parameters
dt = 0.01  # time step
T = 10.0   # total time
N = int(T / dt)

# Angular velocity (constant rotation)
omega_true = np.array([0.5, 0.3, 0.1])  # rad/s

# Process noise (angular velocity uncertainty)
Q = 0.01 * np.eye(3)

# Measurement noise
R_meas = 0.1 * np.eye(3)

# Reference vector in world frame (e.g., gravity)
v_world = np.array([0, 0, 1])

# Simulate true trajectory
R_true = np.zeros((N, 3, 3))
R_true[0] = np.eye(3)  # Start at identity

for i in range(1, N):
    # Add process noise
    omega = omega_true + np.random.multivariate_normal(np.zeros(3), Q)
    # Propagate rotation
    R_true[i] = R_true[i-1] @ exp_so3(omega * dt)

# Generate measurements
measurements = np.zeros((N, 3))
for i in range(N):
    # Rotate reference vector to body frame and add noise
    v_body = R_true[i].T @ v_world
    measurements[i] = v_body + np.random.multivariate_normal(np.zeros(3), R_meas)
    # Normalize (simulate unit vector measurement)
    measurements[i] /= np.linalg.norm(measurements[i])

print(f"Generated {N} time steps of simulation data")

## 3. Invariant EKF Implementation

The IEKF for SO(3) uses right-invariant errors:

- **Error definition**: η = R_true * R_est^T
- **Error dynamics**: η_dot ≈ η * skew(w) where w is noise

This formulation ensures the error dynamics are independent of the current rotation!

In [ ]:
class InvariantEKF:
    def __init__(self, R0, P0, Q, R_meas):
        self.R = R0.copy()  # Rotation estimate
        self.P = P0.copy()  # Covariance in tangent space
        self.Q = Q          # Process noise
        self.R_meas = R_meas  # Measurement noise
    
    def predict(self, omega, dt):
        """Prediction step."""
        # Propagate state
        self.R = self.R @ exp_so3(omega * dt)
        
        # Propagate covariance (simplified)
        # F is approximately identity for small dt
        self.P = self.P + self.Q * dt
    
    def update(self, v_world, v_body_measured):
        """Update step with vector measurement."""
        # Predicted measurement
        v_body_pred = self.R.T @ v_world
        
        # Innovation
        innovation = v_body_measured - v_body_pred
        
        # Measurement Jacobian (in tangent space)
        H = -skew(v_body_pred)
        
        # Kalman gain
        S = H @ self.P @ H.T + self.R_meas
        K = self.P @ H.T @ np.linalg.inv(S)
        
        # Update covariance
        self.P = (np.eye(3) - K @ H) @ self.P
        
        # Compute correction in tangent space
        delta = K @ innovation
        
        # Update state (right multiplication)
        self.R = self.R @ exp_so3(delta)

print("Invariant EKF class defined")

## 4. Run the Filter

In [ ]:
# Initialize filter with noisy initial state
initial_error = np.array([0.3, 0.2, 0.1])
R0 = exp_so3(initial_error)
P0 = 0.1 * np.eye(3)

iekf = InvariantEKF(R0, P0, Q, R_meas)

# Storage for results
R_est = np.zeros((N, 3, 3))
errors = np.zeros(N)

# Run filter
for i in range(N):
    # Predict
    iekf.predict(omega_true, dt)
    
    # Update
    iekf.update(v_world, measurements[i])
    
    # Store results
    R_est[i] = iekf.R
    
    # Compute error
    R_error = R_true[i].T @ R_est[i]
    errors[i] = np.linalg.norm(log_so3(R_error))

print(f"Filter completed. Final error: {errors[-1]:.4f} rad")

## 5. Results and Visualization

In [ ]:
# Plot error over time
time = np.arange(N) * dt

plt.figure(figsize=(12, 4))
plt.plot(time, errors, label='Attitude Error', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Error (rad)')
plt.title('Invariant EKF Performance for Attitude Estimation')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean error: {np.mean(errors):.4f} rad")
print(f"Final error: {errors[-1]:.4f} rad")
print(f"Max error: {np.max(errors):.4f} rad")

## 6. Key Takeaways

1. **Geometric Structure**: IEKF respects the SO(3) structure by working in the tangent space
2. **Invariant Errors**: Errors defined via group operations maintain consistency
3. **Improved Convergence**: The filter quickly converges from initial error
4. **Stability**: The estimate remains accurate throughout the trajectory

## Next Steps

- Try modifying the initial error and process/measurement noise
- Compare with a standard EKF implementation
- Extend to full pose estimation on SE(3)
- Implement multiple vector measurements (accelerometer + magnetometer)

## References

- Bonnabel, S. (2009). "Invariant Extended Kalman Filter: theory and application to a velocity-aided attitude estimation problem."
- [GTSAM EKF Variants Documentation](https://borglab.github.io/gtsam/ekf-variants/)

## Exercise

Try implementing a standard EKF (with errors defined in R^3) and compare its performance with the IEKF. You'll likely observe:
- Less consistent convergence
- Coordinate frame dependence
- Possible numerical issues with rotation matrix orthogonality